In [ ]:
# ===================================================================
# ALPHA PARTICLE INTERACTIONS IN MATTER
# ===================================================================

"""
Alpha particle energy deposition and visible light output in scintillators.

Physics implemented:
✓ Bethe-Bloch stopping power (dE/dx) for heavy charged particles
✓ Range calculations via integration of stopping power
✓ Bragg curves (energy deposition vs depth)
✓ Birks' law for scintillation quenching

Key concepts:
• Alphas are heavy (4 amu), doubly charged (Z=2) particles
• Non-relativistic at typical energies (< 10 MeV)
• Very short range: ~cm in air, ~μm in solids
• High dE/dx → Birks quenching reduces visible light

Applications:
• Alpha spectroscopy with scintillators
• Radon detection
• Radiation damage studies
• Understanding light yield vs particle type
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

print("Alpha Particle Interactions - Notebook Ready!")


In [ ]:
# ===================================================================
# CONFIGURATION - MODIFY THESE VARIABLES
# ===================================================================

DEFAULT_MATERIAL = "plastic_scintillator"
MATERIALS_TO_COMPARE = ["plastic_scintillator", "air", "silicon", "water", "aluminum"]
ALPHA_TEST_ENERGY = 5.5  # MeV (Am-241)
ALPHA_ENERGY_MIN = 0.5
ALPHA_ENERGY_MAX = 10.0
ALPHA_ENERGY_POINTS = 100
ALPHA_QUENCHING_ENERGIES = [1.0, 2.0, 3.0, 4.0, 5.0, 5.5, 6.0, 7.0, 8.0]

from alpha_lib import (
    MATERIALS,
    bethe_bloch_stopping_power,
    calculate_range,
    calculate_visible_energy_birks,
 )

print("✓ Alpha particle library loaded")

In [ ]:
# ===================================================================
# BIRKS' LAW FOR SCINTILLATION QUENCHING
# ===================================================================

"""
In scintillators, heavy particles (alphas) produce less light per MeV
than minimum ionizing particles (electrons) due to quenching.

Birks' Law:
    dL/dx = L0 x (dE/dx) / (1 + kB x dE/dx)

Where:
    dL/dx = light yield per unit length
    L0 = intrinsic light yield (photons/MeV)
    dE/dx = stopping power
    kB = Birks constant (material dependent)

For plastic scintillators: kB ~= 0.1-0.15 g/MeV/cm^2 for alphas
"""

# Calculate quenching for various alpha energies
print("="*75)
print("🔬 BIRKS QUENCHING IN PLASTIC SCINTILLATOR")
print("="*75)

material = DEFAULT_MATERIAL
alpha_energies = np.array(ALPHA_QUENCHING_ENERGIES)

print(f"\nMaterial: {MATERIALS[material]['name']}")
print(f"Birks constant: {MATERIALS[material]['kB']} g/MeV/cm²")
print()
print(f"{'E_alpha (MeV)':<15} {'E_visible (MeV)':<18} {'Quenching':<12} {'Range (μm)':<12}")
print("-"*75)

for E in alpha_energies:
    E_vis, E_dep, q_factor, _, _, _ = calculate_visible_energy_birks(E, material)
    range_cm, _, _ = calculate_range(E, Z_particle=2, material_key=material)
    
    print(f"{E:<15.1f} {E_vis:<18.3f} {q_factor:<12.3f} {range_cm*1e4:<12.1f}")

print("-"*75)
print()
print("💡 Quenching factor = E_visible / E_deposited")
print("   • Lower energy alphas have higher dE/dx → more quenching")
print("   • Typical: alphas give ~10× less light than electrons at same energy")
print("="*75)

In [ ]:
# ===================================================================
# COMPREHENSIVE ALPHA PARTICLE ANALYSIS PLOTS
# ===================================================================

material = DEFAULT_MATERIAL
mat = MATERIALS[material]

# Energy range for plots
alpha_energies = np.linspace(ALPHA_ENERGY_MIN, ALPHA_ENERGY_MAX, ALPHA_ENERGY_POINTS)

# Calculate properties for all energies
ranges = []
stopping_powers = []
visible_energies = []
quenching_factors = []

for E in alpha_energies:
    # Range
    R, _, _ = calculate_range(E, Z_particle=2, material_key=material)
    ranges.append(R * 1e4)  # Convert to μm
    
    # Stopping power at initial energy
    dEdx = bethe_bloch_stopping_power(E, Z_particle=2, material_key=material)
    stopping_powers.append(dEdx)
    
    # Visible energy with Birks quenching
    E_vis, _, q_fac, _, _, _ = calculate_visible_energy_birks(E, material)
    visible_energies.append(E_vis)
    quenching_factors.append(q_fac)

ranges = np.array(ranges)
stopping_powers = np.array(stopping_powers)
visible_energies = np.array(visible_energies)
quenching_factors = np.array(quenching_factors)

# Create comprehensive figure
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Plot 1: Stopping power vs energy
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(alpha_energies, stopping_powers, linewidth=2.5, color=mat['color'])
ax1.set_xlabel('Alpha Energy (MeV)', fontsize=11)
ax1.set_ylabel('Stopping Power dE/dx (MeV/cm)', fontsize=11)
ax1.set_title('Stopping Power in ' + mat['name'], fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, ALPHA_ENERGY_MAX)

# Plot 2: Range vs energy
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(alpha_energies, ranges, linewidth=2.5, color=mat['color'])
ax2.set_xlabel('Alpha Energy (MeV)', fontsize=11)
ax2.set_ylabel('Range (μm)', fontsize=11)
ax2.set_title('Alpha Range in ' + mat['name'], fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, ALPHA_ENERGY_MAX)

# Plot 3: Visible energy vs deposited energy
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(alpha_energies, alpha_energies, '--', linewidth=2, 
         color='gray', alpha=0.6, label='No quenching (ideal)')
ax3.plot(alpha_energies, visible_energies, linewidth=2.5, 
         color='#F18F01', label='With Birks quenching')
ax3.set_xlabel('Alpha Energy (Deposited) [MeV]', fontsize=11)
ax3.set_ylabel('Visible Energy (MeV)', fontsize=11)
ax3.set_title('Scintillation Light Output', fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, ALPHA_ENERGY_MAX)
ax3.set_ylim(0, ALPHA_ENERGY_MAX)

# Plot 4: Quenching factor vs energy
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(alpha_energies, quenching_factors, linewidth=2.5, color='#A23B72')
ax4.axhline(y=1.0, linestyle=':', color='gray', alpha=0.6, label='No quenching')
ax4.set_xlabel('Alpha Energy (MeV)', fontsize=11)
ax4.set_ylabel('Quenching Factor (E_visible / E_deposited)', fontsize=11)
ax4.set_title('Birks Quenching Factor', fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(0, ALPHA_ENERGY_MAX)
ax4.set_ylim(0, 1.1)

# Plot 5 & 6: Bragg curve for specific alpha
E_alpha = ALPHA_TEST_ENERGY
E_vis, E_dep, q_fac, energies, depths, dEdx_track = calculate_visible_energy_birks(
    E_alpha, material, num_steps=500)

depths_um = depths * 1e4  # Convert to μm

ax5 = fig.add_subplot(gs[2, 0])
ax5.plot(depths_um, dEdx_track, linewidth=2.5, color='#2E86AB')
ax5.set_xlabel('Depth (μm)', fontsize=11)
ax5.set_ylabel('dE/dx (MeV/cm)', fontsize=11)
ax5.set_title(f'Bragg Curve: {E_alpha} MeV Alpha in {mat["name"]}', fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.axvline(x=depths_um[-1], linestyle='--', color='red', alpha=0.6, 
            label=f'Range = {depths_um[-1]:.1f} μm')
ax5.legend(fontsize=10)

ax6 = fig.add_subplot(gs[2, 1])
# Calculate cumulative visible energy
dE = np.abs(np.diff(energies))
dEdx_mass = dEdx_track[:-1] / mat['density']
birks_factor = 1.0 / (1.0 + mat['kB'] * dEdx_mass)
dE_visible = dE * birks_factor
cumulative_visible = np.concatenate([[0], np.cumsum(dE_visible)])

ax6.plot(depths_um, energies, linewidth=2.5, color='#F18F01', label='Deposited energy')
ax6.plot(depths_um, cumulative_visible, linewidth=2.5, color='#2E86AB', 
         linestyle='--', label='Visible energy (cumulative)')
ax6.set_xlabel('Depth (μm)', fontsize=11)
ax6.set_ylabel('Energy (MeV)', fontsize=11)
ax6.set_title(f'Energy Deposition: {E_alpha} MeV Alpha', fontweight='bold')
ax6.legend(fontsize=10)
ax6.grid(True, alpha=0.3)

plt.suptitle(f'Alpha Particle Interactions in {mat["name"]}', 
             fontsize=14, fontweight='bold', y=0.995)

plt.show()

# Print summary
print(f"\n{'='*75}")
print(f"📊 SUMMARY FOR {E_alpha} MeV ALPHA IN {mat['name'].upper()}")
print(f"{'='*75}")
print(f"Range:            {depths_um[-1]:.1f} μm")
print(f"Deposited energy: {E_dep:.2f} MeV")
print(f"Visible energy:   {E_vis:.2f} MeV")
print(f"Quenching factor: {q_fac:.3f}")
print(f"Light reduction:  {(1-q_fac)*100:.1f}% less than electrons")
print(f"{'='*75}")

In [ ]:
# ===================================================================
# MATERIAL COMPARISON FOR ALPHA PARTICLES
# ===================================================================

E_alpha = ALPHA_TEST_ENERGY

# Calculate for all materials
materials_to_compare = MATERIALS_TO_COMPARE

print("="*85)
print(f"📊 ALPHA PARTICLE ({E_alpha} MeV) IN DIFFERENT MATERIALS")
print("="*85)
print()
print(f"{'Material':<25} {'Density':<12} {'dE/dx':<15} {'Range':<15} {'E_visible':<12}")
print(f"{'':25} {'(g/cm³)':<12} {'(MeV/cm)':<15} {'(μm)':<15} {'(MeV)':<12}")
print("-"*85)

for mat_key in materials_to_compare:
    mat = MATERIALS[mat_key]
    
    # Stopping power at initial energy
    dEdx = bethe_bloch_stopping_power(E_alpha, Z_particle=2, material_key=mat_key)
    
    # Range
    range_cm, _, _ = calculate_range(E_alpha, Z_particle=2, material_key=mat_key)
    range_um = range_cm * 1e4
    
    # Visible energy (only for scintillators)
    if mat['kB'] > 0:
        E_vis, _, q_fac, _, _, _ = calculate_visible_energy_birks(E_alpha, mat_key)
        E_vis_str = f"{E_vis:.2f}"
    else:
        E_vis_str = "N/A"
    
    print(f"{mat['name']:<25} {mat['density']:<12.3f} {dEdx:<15.2f} "
          f"{range_um:<15.1f} {E_vis_str:<12}")

print("-"*85)
print()
print("💡 Key observations:")
print("   • Range inversely proportional to density")
print("   • Air: ~cm range, Silicon: ~μm range")
print("   • Only scintillators produce visible light")
print("   • Plastic scintillator: ~70% light reduction due to quenching")
print("="*85)

# Comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Range comparison
materials_plot = []
ranges_plot = []
colors_plot = []
for mat_key in materials_to_compare:
    mat = MATERIALS[mat_key]
    range_cm, _, _ = calculate_range(E_alpha, Z_particle=2, material_key=mat_key)
    materials_plot.append(mat['name'])
    ranges_plot.append(range_cm * 1e4)  # μm
    colors_plot.append(mat['color'])

ax1.barh(materials_plot, ranges_plot, color=colors_plot, alpha=0.8)
ax1.set_xlabel('Range (μm)', fontsize=11)
ax1.set_title(f'{E_alpha} MeV Alpha Range Comparison', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.set_xscale('log')

# Stopping power comparison
energies = np.linspace(ALPHA_ENERGY_MIN, ALPHA_ENERGY_MAX, ALPHA_ENERGY_POINTS)
for mat_key in materials_to_compare:
    mat = MATERIALS[mat_key]
    dEdx_arr = bethe_bloch_stopping_power(energies, Z_particle=2, material_key=mat_key)
    ax2.plot(energies, dEdx_arr, linewidth=2.5, label=mat['name'], 
             color=mat['color'], alpha=0.8)

ax2.set_xlabel('Alpha Energy (MeV)', fontsize=11)
ax2.set_ylabel('Stopping Power dE/dx (MeV/cm)', fontsize=11)
ax2.set_title('Stopping Power Comparison', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, ALPHA_ENERGY_MAX)

plt.tight_layout()
plt.show()

In [ ]:
# ===================================================================
# PRACTICAL APPLICATION: ALPHA SPECTROSCOPY WITH SCINTILLATOR
# ===================================================================

"""
Simulate what an alpha spectrum looks like in a plastic scintillator
detector, accounting for Birks quenching.

Common alpha sources:
• Am-241: 5.486 MeV (85%), 5.443 MeV (13%)
• Pu-239: 5.156 MeV (73%), 5.144 MeV (15%), 5.105 MeV (12%)
• Ra-226: 4.784 MeV (94%)
• Rn-222: 5.490 MeV (100%)
"""

# Simulate Am-241 spectrum
material = DEFAULT_MATERIAL

# Am-241 alpha lines
am241_lines = [
    {'energy': 5.486, 'intensity': 0.851, 'label': '⁵.⁴⁸⁶ MeV'},
    {'energy': 5.443, 'intensity': 0.129, 'label': '⁵.⁴⁴³ MeV'},
]

# Pu-239 alpha lines
pu239_lines = [
    {'energy': 5.156, 'intensity': 0.731, 'label': '⁵.¹⁵⁶ MeV'},
    {'energy': 5.144, 'intensity': 0.151, 'label': '⁵.¹⁴⁴ MeV'},
    {'energy': 5.105, 'intensity': 0.118, 'label': '⁵.¹⁰⁵ MeV'},
]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot Am-241
ax1, ax2 = axes[0]

# True energy spectrum
for line in am241_lines:
    E_true = line['energy']
    intensity = line['intensity']
    ax1.axvline(E_true, linewidth=3, alpha=0.7, 
                label=f"{line['label']} ({intensity*100:.1f}%)")

ax1.set_xlabel('Alpha Energy (MeV)', fontsize=11)
ax1.set_ylabel('Intensity (arb.)', fontsize=11)
ax1.set_title('Am-241: True Alpha Spectrum', fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(5.3, 5.6)
ax1.set_ylim(0, 1.2)

# Visible energy spectrum (with quenching)
for line in am241_lines:
    E_true = line['energy']
    intensity = line['intensity']
    E_vis, _, q_fac, _, _, _ = calculate_visible_energy_birks(E_true, material)
    ax2.axvline(E_vis, linewidth=3, alpha=0.7, 
                label=f"{E_vis:.3f} MeV (q={q_fac:.3f})")

ax2.set_xlabel('Visible Energy (MeV)', fontsize=11)
ax2.set_ylabel('Intensity (arb.)', fontsize=11)
ax2.set_title('Am-241: Scintillator Response (with Birks quenching)', fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(3.5, 4.0)
ax2.set_ylim(0, 1.2)

# Plot Pu-239
ax3, ax4 = axes[1]

# True energy spectrum
for line in pu239_lines:
    E_true = line['energy']
    intensity = line['intensity']
    ax3.axvline(E_true, linewidth=3, alpha=0.7, 
                label=f"{line['label']} ({intensity*100:.1f}%)")

ax3.set_xlabel('Alpha Energy (MeV)', fontsize=11)
ax3.set_ylabel('Intensity (arb.)', fontsize=11)
ax3.set_title('Pu-239: True Alpha Spectrum', fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(5.0, 5.3)
ax3.set_ylim(0, 1.2)

# Visible energy spectrum (with quenching)
for line in pu239_lines:
    E_true = line['energy']
    intensity = line['intensity']
    E_vis, _, q_fac, _, _, _ = calculate_visible_energy_birks(E_true, material)
    ax4.axvline(E_vis, linewidth=3, alpha=0.7, 
                label=f"{E_vis:.3f} MeV (q={q_fac:.3f})")

ax4.set_xlabel('Visible Energy (MeV)', fontsize=11)
ax4.set_ylabel('Intensity (arb.)', fontsize=11)
ax4.set_title('Pu-239: Scintillator Response (with Birks quenching)', fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(3.3, 3.7)
ax4.set_ylim(0, 1.2)

plt.suptitle('Alpha Spectroscopy: True vs Scintillator Response', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Print calibration info
print("="*75)
print("🔬 ALPHA SPECTROSCOPY CALIBRATION")
print("="*75)
print(f"\nMaterial: {MATERIALS[material]['name']}")
print(f"Birks constant: {MATERIALS[material]['kB']} g/MeV/cm²")
print()
print("Am-241:")
for line in am241_lines:
    E_true = line['energy']
    E_vis, _, q_fac, _, _, _ = calculate_visible_energy_birks(E_true, material)
    print(f"  {E_true:.3f} MeV → {E_vis:.3f} MeV visible (q={q_fac:.3f})")

print()
print("Pu-239:")
for line in pu239_lines:
    E_true = line['energy']
    E_vis, _, q_fac, _, _, _ = calculate_visible_energy_birks(E_true, material)
    print(f"  {E_true:.3f} MeV → {E_vis:.3f} MeV visible (q={q_fac:.3f})")

print()
print("💡 Key points:")
print("   • Alpha peaks shift to ~70% of true energy")
print("   • Peak separation preserved (for particle ID)")
print("   • Must calibrate with known alpha source")
print("   • Lower energy alphas are more quenched")
print("="*75)